In [1]:
import os
import sys
import pickle
import numpy as np
from numba import njit
import itertools as itt
import aerosandbox as asb

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from Aircraft.Planform import Planform
from Aircraft.Fixed import Fixed
from Drag.Fuselage import Fuselage
from Drag.Bay import Bay
from Drag.LandingGear import LandingGear
from Aircraft.Aircraft import Aircraft
from global_parameters import Assumptions
from Requirements.FuelReq import FuelReq
from Requirements.LGReq import LGReq
from Requirements.MassReq import MassReq
from Requirements.MDReq import MDReq
from Requirements.EmpennageReq import EmpennageReq
from Requirements.Requirement import Requirement
from EmpennageSizing.TailFinder import TailFinder
from EmpennageSizing.CanardFinder import CanardFinder

from structural_analysis.Material import Material
from structural_analysis.iterative_planform_sizing import size_planform

In [2]:
assumptions = Assumptions()

#TODO load the fuselage here
with open("pickles/fixed_pickle.pcl", "rb") as f:
    fixed:Fixed = pickle.load(f)

for component in fixed.drag_components(False):
    component.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
    component.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
    component.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
for component in fixed.drag_components(True):
    component.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)

# Defining the standard aircraft with the standard planform
To be used when ppl don't wanna build their own planform

In [3]:
standard_wing = Planform(aspect_ratio=27, span=2.667, sweep_quarter_deg=15., taper=.5, thickness_to_chord=0.12, cm_quarter_chord=0,
                         wetted_surface_ratio=1.07, interference_factor=1.0, clmax=1.25, flap=False)

In [4]:
assumptions = Assumptions()

go_around_atmosphere = asb.Atmosphere(assumptions.altitude_go_round)
sea_level_atmosphere = asb.Atmosphere()

standard_wing.add_cache_entry('cruise', assumptions.mach_cruise, assumptions.altitude_cruise)
standard_wing.add_cache_entry('mach_max', assumptions.mach_max, assumptions.altitude_mach_max)
#NOTE: not fully technically correct but prevents coupling which would be problematic, acceptable as CD0 dept. on mach is small @ low mach
standard_wing.add_cache_entry('go_around', assumptions.airspeed_approach / go_around_atmosphere.speed_of_sound(), assumptions.altitude_go_round)
standard_wing.add_cache_entry('takeoff', assumptions.airspeed_approach / sea_level_atmosphere.speed_of_sound(), 0.)

In [5]:
material_skin = Material(assumptions.cfrp_density, elastic_modulus=assumptions.cfrp_Young_modulus, 
                         poisson_ratio=assumptions.cfrp_poisson, shear_modulus=assumptions.cfrp_Young_modulus / 2 / (1 + assumptions.cfrp_poisson),
                         yield_strength=assumptions.cfrp_yield_strength, fracture_strength=assumptions.cfrp_yield_strength)

fuselage_diameter = fixed.fuselage.diameter_max
size_planform(planform=standard_wing, thicknesses=assumptions.allowable_thicknesses, fuselage_diameter=fuselage_diameter, material_skin=material_skin, density_core=assumptions.foam_denisty)
    

154.42027952683736
Stresses 259356790.75382066, 3669669268.6498384, 0.0004
154.42027952683736
Stresses 129458939.17300315, 1943028761.8056252, 0.0007999999999999999
154.42027952683736
Stresses 86159655.31273068, 1372817768.47877, 0.0012
154.42027952683736
Stresses 64510013.38259439, 1092009789.1343799, 0.0015999999999999999
154.42027952683736
Stresses 51520228.22451267, 927206098.0908614, 0.002
154.42027952683736
Stresses 42860371.45245818, 820609083.783128, 0.0024000000000000002
154.42027952683736
Stresses 36674759.472419284, 747445778.1340951, 0.0028
154.42027952683736
Stresses 32035550.487390045, 695322380.3516374, 0.0031999999999999997
154.42027952683736
Stresses 28427276.83236736, 657340941.7254019, 0.0036
154.42027952683736
Stresses 25540657.90834918, 629344423.9535263, 0.004
154.42027952683736
Stresses 23178878.788697958, 608662733.9813275, 0.0044
154.42027952683736
Stresses 21210729.52232194, 593483965.5803328, 0.0048
154.42027952683736
Stresses 19545372.450772993, 582514083.40

In [6]:
print(standard_wing.mass_cache, standard_wing.x_cg_cache)

2.4014122912631892 0.21733709563463582


# We consider the thing to be tailed

In [7]:
tail = TailFinder(fixed, material=material_skin, core_density=assumptions.foam_denisty, thicknesses=assumptions.allowable_thicknesses, safety_factor=assumptions.structural_safety_factor, AR_h=7., taper_h=.7, taper_v=.8).find_planforms(standard_wing)

print(tail[0].wing_area / standard_wing.wing_area)

9.276369401496753
Stresses 27027356.32289794, 126614710.9638846, 0.0004
55.50293971559812
Stresses 601317669.5955884, 13915654442.58516, 0.0004
55.50293971559812
Stresses 300642031.2134073, 10429991727.097475, 0.0007999999999999999
55.50293971559812
Stresses 200416818.41934696, 10134010888.349625, 0.0012
55.50293971559812
Stresses 150304212.02231663, 9712907069.611082, 0.0015999999999999999
55.50293971559812
Stresses 120236648.1840986, 7845142315.794179, 0.002
55.50293971559812
Stresses 100191605.62528646, 5346772925.430229, 0.0024000000000000002
55.50293971559812
Stresses 85873718.08327776, 3420453615.7153463, 0.0028
55.50293971559812
Stresses 75135302.42677134, 2220684419.399443, 0.0031999999999999997
55.50293971559812
Stresses 66783201.3605997, 1505327066.5113502, 0.0036
55.50293971559812
Stresses 60101520.50766235, 1071206009.6707945, 0.004
55.50293971559812
Stresses 54634690.718895346, 797943423.541804, 0.0044
55.50293971559812
Stresses 50078999.22825628, 619790111.071658, 0.0048


In [8]:
for t in tail:
    t.add_cache_entry('cruise', assumptions.mach_cruise, assumptions.altitude_cruise)
    t.add_cache_entry('mach_max', assumptions.mach_max, assumptions.altitude_mach_max)
#NOTE: not fully technically correct but prevents coupling which would be problematic, acceptable as CD0 dept. on mach is small @ low mach
    t.add_cache_entry('go_around', assumptions.airspeed_approach / go_around_atmosphere.speed_of_sound(), assumptions.altitude_go_round)
    t.add_cache_entry('takeoff', assumptions.airspeed_approach / sea_level_atmosphere.speed_of_sound(), 0.)

In [9]:
ac = Aircraft(fixed, [standard_wing] + tail)


In [10]:
for acp in ac.planforms:
    print(acp.mass_cache)

print(ac.fixed.x_cg_min, ac.fixed.x_cg_max, ac.fixed.x_LE_wing)

2.4014122912631892
0.010341167564948486
0.0325302011393337
1.0317893591765872 1.09066819465422 1.3150000000000002


# Requirement check for the aircraft

In [11]:
requirements:list[Requirement] = [
    MassReq(50.),
    MDReq(),
    FuelReq(),
    LGReq(),
    EmpennageReq(),
]

requirement_labels = [
    "MTOM",
    "Matching Diagram",
    "Fuel",
    "Landing Gear",
    "Empennage Requirement"
]

In [12]:
failed_reqs = list()
for requirement, label in zip(requirements, requirement_labels):
    if not requirement.assess(ac):
        failed_reqs.append(label)

if len(failed_reqs):
    print(f"ac mass: {ac.total_mass()}, {ac.planforms[0].oswald}")
    print(f"MainWing: AR={ac.planforms[0].aspect_ratio}, tc={ac.planforms[0].thickness_to_chord}, sweep={np.rad2deg(ac.planforms[0].sweep_quarter_rad)} deg, cmac={ac.planforms[0].cm_quarter_chord}")
    print(f"Failed: {failed_reqs}")
    print()

Fuel available: 11.000000001438 kg
Fuel required: 5.295781860578281 kg
Difference: 5.704218140859719 kg
all constraints satisfied
ac mass: 36.508444665772416, 0.680337246179267
MainWing: AR=27, tc=0.12, sweep=14.999999999999998 deg, cmac=0
Failed: ['Empennage Requirement']



In [13]:
#TODO: ctrl surface sizing